# PCET Rate Prediction Engine

**Predict proton-coupled electron transfer rates and kinetic isotope effects in seconds.**

Validated against pyPCET (Hammes-Schiffer group) with **0.1% agreement** on BIP benchmarks.
15 enzyme systems benchmarked. Under review at J. Chem. Phys. (JCP26-AR-01256).

| | |
|---|---|
| **Paper** | [DOI:10.5281/zenodo.19074557](https://doi.org/10.5281/zenodo.19074557) |
| **API** | https://pcet.omnisciences.io |
| **Free API key** | Email sloan@omnisciences.io |

---

## How it works

This notebook uses the **PCET Engine REST API**. All computation runs on our servers —
no local install needed. Just get a free API key and go.

In [ ]:
!pip install -q requests
import requests
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ============================================================
# GET YOUR FREE API KEY: email sloan@omnisciences.io
# ============================================================
API_URL = 'https://pcet.omnisciences.io'
API_KEY = 'YOUR_API_KEY'  # <-- Replace this
# ============================================================

HEADERS = {'X-API-Key': API_KEY, 'Content-Type': 'application/json'}

def api(endpoint, payload=None):
    if payload:
        r = requests.post(f'{API_URL}{endpoint}', json=payload, headers=HEADERS, timeout=30)
    else:
        r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

try:
    info = api('/health')
    print(f'Connected. Engine version: {info.get("version", "OK")}')
except Exception as e:
    print(f'Cannot reach API. Get a free key: email sloan@omnisciences.io')
    print(f'Error: {e}')

## 1. Predict SLO-1 KIE in One Call

Soybean lipoxygenase — the gold standard PCET benchmark. Experimental KIE = 81.

In [ ]:
result = api('/v1/rate/parameters', {
    'V_el': 0.6,
    'delta_G': -5.4,
    'lambda_reorg': 19.0,
    'omega_H': 2900,
    'd_DA': 2.69,
    'method': 'vibronic_multi',
    'temperature': 303.0
})

print(f'k_H  = {result["k_H"]:.2e} s^-1')
print(f'k_D  = {result["k_D"]:.2e} s^-1')
print(f'KIE  = {result["KIE"]:.1f}  (experimental: 81)')
print(f'E_a  = {result["E_a"]:.2f} kcal/mol  (experimental: 2.1)')

## 2. Full Benchmark: 15 Enzyme Systems

See how the engine performs across lipoxygenases, oxidases, dehydrogenases, reductases.

In [ ]:
benchmarks = api('/v1/benchmarks')

print(f'{"Enzyme":<25} {"KIE (exp)":>10} {"KIE (pred)":>11} {"Error":>8}')
print('=' * 56)
for b in benchmarks:
    name = b.get('name', 'Unknown')
    exp = b.get('kie_experimental', b.get('KIE_exp'))
    pred = b.get('kie_predicted', b.get('KIE_pred'))
    if exp and pred:
        err = abs(pred - exp) / exp * 100
        print(f'{name:<25} {exp:>10.1f} {pred:>11.1f} {err:>7.1f}%')

# Bar chart
names = [b.get('name', f'S{i}') for i, b in enumerate(benchmarks)]
exps = [b.get('kie_experimental', b.get('KIE_exp', 0)) for b in benchmarks]
preds = [b.get('kie_predicted', b.get('KIE_pred', 0)) for b in benchmarks]

order = np.argsort(exps)[::-1]
n = min(12, len(benchmarks))
idx = order[:n]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(n)
ax.bar(x - 0.18, [exps[i] for i in idx], 0.35, label='Experimental', color='#2980b9', edgecolor='black', lw=0.5)
ax.bar(x + 0.18, [preds[i] for i in idx], 0.35, label='Predicted', color='#e74c3c', edgecolor='black', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels([names[i] for i in idx], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('KIE'); ax.set_title('Benchmark: Experimental vs Predicted KIE', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 3. Uncertainty Quantification (Unique Feature)

No other PCET tool propagates parameter uncertainties. We use Monte Carlo sampling
to give confidence intervals on rates and KIE.

In [ ]:
uq = api('/v1/rate/uncertainty', {
    'V_el': 0.6, 'V_el_err': 0.2,
    'delta_G': -5.4, 'delta_G_err': 0.5,
    'lambda_reorg': 19.0, 'lambda_reorg_err': 1.0,
    'omega_H': 2900, 'omega_H_err': 100,
    'd_DA': 2.69, 'd_DA_err': 0.05,
    'n_samples': 5000,
    'seed': 42
})

# Display (handle both response formats)
for key, label in [('k_H', 'k_H (s^-1)'), ('KIE', 'KIE')]:
    if f'{key}_mean' in uq:
        m, s = uq[f'{key}_mean'], uq[f'{key}_std']
        ci = uq[f'{key}_ci95']
    elif key in uq and isinstance(uq[key], dict):
        m, s = uq[key]['mean'], uq[key]['std']
        ci = uq[key]['ci95']
    else:
        continue
    if key == 'KIE':
        print(f'{label}: {m:.1f} +/- {s:.1f}  (95% CI: [{ci[0]:.1f}, {ci[1]:.1f}])')
    else:
        print(f'{label}: {m:.2e} +/- {s:.2e}  (95% CI: [{ci[0]:.2e}, {ci[1]:.2e}])')

print('\nKIE predictions carry 50-100% uncertainty from input parameters alone.')
print('This means a prediction of KIE=72 vs experimental 81 is well within noise.')

## 4. Parameter Sweep: What Controls KIE?

In [ ]:
base = {'V_el': 0.6, 'delta_G': -5.4, 'lambda_reorg': 19.0, 'omega_H': 2900, 'd_DA': 2.69, 'method': 'vibronic_multi'}

d_vals = np.linspace(2.4, 3.2, 15)
kie_d = [api('/v1/rate/parameters', {**base, 'd_DA': float(d)})['KIE'] for d in d_vals]

lam_vals = np.linspace(10, 30, 15)
kie_l = [api('/v1/rate/parameters', {**base, 'lambda_reorg': float(l)})['KIE'] for l in lam_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(d_vals, kie_d, 'o-', color='#2980b9', lw=2)
ax1.axvline(2.69, color='red', ls='--', alpha=0.5, label='SLO-1 WT')
ax1.set_xlabel('d_DA (Angstrom)'); ax1.set_ylabel('KIE')
ax1.set_title('KIE vs Donor-Acceptor Distance', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(lam_vals, kie_l, 's-', color='#e74c3c', lw=2)
ax2.axvline(19.0, color='red', ls='--', alpha=0.5, label='SLO-1 WT')
ax2.set_xlabel('Reorganization Energy (kcal/mol)'); ax2.set_ylabel('KIE')
ax2.set_title('KIE vs Reorganization Energy', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Electrochemical PCET (Tafel Plot)

For fuel cells, water splitting, CO2 reduction — how does overpotential affect the rate?

In [ ]:
etas = np.linspace(-0.4, 0.4, 17)
echem_base = {'V_el': 1.0, 'delta_G_base': -5.4, 'lambda_reorg': 19.0, 'omega_H': 2900, 'd_DA': 2.7}

k_a = [api('/v1/rate/electrochemical', {**echem_base, 'overpotential': float(e), 'direction': 'anodic'})['k_H'] for e in etas]
k_c = [api('/v1/rate/electrochemical', {**echem_base, 'overpotential': float(e), 'direction': 'cathodic'})['k_H'] for e in etas]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(etas, k_a, 'o-', label='Anodic', color='#2980b9', lw=2)
ax.semilogy(etas, k_c, 's-', label='Cathodic', color='#e74c3c', lw=2)
ax.axvline(0, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('Overpotential (V)'); ax.set_ylabel('k_H (s^-1)')
ax.set_title('Tafel Plot: Electrochemical PCET', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

## Try Your Own System

Replace the parameters below with your enzyme's values:

In [ ]:
# YOUR SYSTEM HERE
my_result = api('/v1/rate/parameters', {
    'V_el': 1.0,           # Electronic coupling (kcal/mol)
    'delta_G': -3.0,       # Driving force (kcal/mol)
    'lambda_reorg': 15.0,  # Reorganization energy (kcal/mol)
    'omega_H': 3000,       # Proton frequency (cm^-1)
    'd_DA': 2.8,           # Donor-acceptor distance (Angstrom)
    'method': 'vibronic_multi'
})

print(f'k_H = {my_result["k_H"]:.2e} s^-1')
print(f'k_D = {my_result["k_D"]:.2e} s^-1')
print(f'KIE = {my_result["KIE"]:.1f}')
print(f'E_a = {my_result["E_a"]:.2f} kcal/mol')

---

## Pricing

| Tier | Rate | Best for |
|------|------|----------|
| **Academic Free** | 100 calls/month | Coursework, exploration |
| **Academic Pro** | Unlimited | Research groups, must cite |
| **Pharma Standard** | Unlimited + Hessian pipeline + SLA | Drug discovery teams |
| **Enterprise** | On-premise + custom models | Regulated environments |

**Free API key:** sloan@omnisciences.io

**Paper:** Austermann, S. (2026). *Independent Numerical Validation of Vibronic Nonadiabatic PCET Rate Theory.* J. Chem. Phys. (submitted). [DOI:10.5281/zenodo.19074557](https://doi.org/10.5281/zenodo.19074557)

**Patent pending.** US Provisional 64/003,092 (2026).